# Universal Document Ingestion for RAG with Pinecone

This notebook provides a complete pipeline for ingesting various document types (`.pdf`, `.docx`, `.xlsx`, `.txt`, `.xml`, `.jpg`, `.png`), processing their content, generating embeddings, and upserting them into a Pinecone vector index. It is designed to handle both text-based and scanned (image-based) documents automatically.

### 1. Setup: Install Dependencies

**Install Tesseract-OCR**

Tesseract is an optical character recognition (OCR) engine required for extracting text from images. Please install it on your system.

* **Windows**:
    Download and run the installer from the [Tesseract at UB Mannheim](https://github.com/UB-Mannheim/tesseract/wiki) page. During installation, make sure to note the installation path. You may need to add this path (e.g., `C:\Program Files\Tesseract-OCR`) to your system's `PATH` environment variable.

* **macOS**:
    You can use Homebrew to install Tesseract:
    ```bash
    brew install tesseract
    ```

* **Linux (Debian/Ubuntu)**:
    ```bash
    sudo apt-get update
    sudo apt-get install tesseract-ocr
    ```

### 2. Environment Configuration

**Create a `.env` file** in the same directory as this notebook. This file will store your secret API keys. Add the following content to it, replacing the placeholder values with your actual credentials:

```
PINECONE_API_KEY="YOUR_PINECONE_API_KEY_HERE"
PINECONE_INDEX_NAME="your-pinecone-index-name"
GEMINI_API_KEY="YOUR_GEMINI_API_KEY_HERE"
```

In [ ]:
import os
import io
from pathlib import Path
from typing import List
import xml.etree.ElementTree as ET
from dotenv import load_dotenv

# Data Handling & Processing
import fitz  # PyMuPDF
import docx
import pandas as pd
from PIL import Image
import pytesseract
from langchain.text_splitter import RecursiveCharacterTextSplitter
from tqdm.auto import tqdm

# AI & Vector DB
import torch
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec
import google.generativeai as genai

# Load environment variables from .env file
load_dotenv()

# Load API keys and environment variables
try:
    PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')
    PINECONE_INDEX_NAME = os.getenv('PINECONE_INDEX_NAME')
    GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

    if not all([PINECONE_API_KEY, PINECONE_INDEX_NAME, GEMINI_API_KEY]):
        raise ValueError("One or more environment variables (PINECONE_API_KEY, PINECONE_INDEX_NAME, GEMINI_API_KEY) are missing.")

    # Configure the Generative AI client
    genai.configure(api_key=GEMINI_API_KEY)
    print("API keys and environment variables loaded successfully.")

except Exception as e:
    print(f"Error loading environment variables or configuring API: {e}")

### 3. Initialize Pinecone Index and Embedding Model

In [ ]:
# --- Configuration ---
MODEL_NAME = "all-mpnet-base-v2"
EMBEDDING_DIMENSIONS = 768  # Fixed for the chosen model
PINECONE_CLOUD = 'aws' # Or 'gcp', 'azure'
PINECONE_REGION = 'us-east-1'

# --- Initialize Pinecone ---
try:
    print("Initializing Pinecone client...")
    pc = Pinecone(api_key=PINECONE_API_KEY)
    index_name = PINECONE_INDEX_NAME

    if index_name not in pc.list_indexes().names():
        print(f"Index '{index_name}' not found. Creating a new serverless index...")
        pc.create_index(
            name=index_name,
            dimension=EMBEDDING_DIMENSIONS,
            metric="cosine",
            spec=ServerlessSpec(
                cloud=PINECONE_CLOUD,
                region=PINECONE_REGION
            )
        )
        print(f"Index '{index_name}' created successfully.")
    else:
        print(f"Index '{index_name}' already exists.")

    index = pc.Index(index_name)
    print("\nInitial Pinecone index stats:")
    print(index.describe_index_stats())

except Exception as e:
    print(f"Error initializing Pinecone: {e}")

# --- Load Embedding Model (Optimized for GPU) ---
try:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\nUsing device: {device}")
    print(f"Loading sentence transformer model: {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME, device=device)
    print("Embedding model loaded successfully.")
except Exception as e:
    print(f"Error loading embedding model: {e}")

### 4. Define Helper Functions for ETL (Extract, Transform, Load)

In [ ]:
def find_all_documents(folder_path: str) -> List[str]:
    """
    Recursively find all compatible documents in a folder and its subfolders.
    """
    supported_extensions = {
        '.pdf', '.docx', '.txt', '.xlsx',
        '.jpg', '.jpeg', '.png', '.xml'
    }
    document_paths = []

    folder = Path(folder_path)
    if not folder.exists():
        print(f"Error: The folder '{folder_path}' does not exist.")
        return document_paths

    for file_path in folder.rglob('*'):
        if (file_path.is_file() and
                file_path.suffix.lower() in supported_extensions and
                not file_path.name.startswith('._') and
                '__MACOSX' not in file_path.parts):
            document_paths.append(str(file_path))

    return sorted(document_paths)

def extract_text_from_document(file_path: str) -> List[dict]:
    """
    Extracts text from various document types using the best method for each.
    """
    file_extension = Path(file_path).suffix.lower()
    print(f"-> Extracting content from: {Path(file_path).name}")

    pages = []

    try:
        if file_extension == '.pdf':
            doc = fitz.open(file_path)
            for i, page in enumerate(tqdm(doc, desc="Processing PDF pages", leave=False)):
                text = page.get_text()
                # If text is minimal, assume it's a scanned page and run OCR
                if len(text.strip()) < 150:
                    try:
                        pix = page.get_pixmap(dpi=300)
                        img_data = pix.tobytes("png")
                        image = Image.open(io.BytesIO(img_data))
                        ocr_text = pytesseract.image_to_string(image)
                        text = ocr_text
                    except Exception as e:
                        print(f"   - OCR failed on page {i+1}: {e}, skipping page.")
                        continue
                if text:
                    pages.append({"page_number": i + 1, "content": text})

        elif file_extension == '.docx':
            doc = docx.Document(file_path)
            full_text = "\n".join([para.text for para in doc.paragraphs])
            if full_text:
                pages.append({"page_number": 1, "content": full_text})

        elif file_extension == '.txt':
            with open(file_path, 'r', encoding='utf-8') as f:
                full_text = f.read()
            if full_text:
                pages.append({"page_number": 1, "content": full_text})

        elif file_extension == '.xlsx':
            df_map = pd.read_excel(file_path, sheet_name=None, header=None)
            for i, (sheet_name, df) in enumerate(df_map.items()):
                sheet_text = df.to_string(index=False, header=False)
                cleaned_text = "\n".join([line for line in sheet_text.split('\n') if line.strip()])
                if cleaned_text:
                    content = f"--- Content from Sheet: {sheet_name} ---\n\n{cleaned_text}"
                    pages.append({"page_number": i + 1, "content": content})

        elif file_extension in ['.jpg', '.jpeg', '.png']:
            image = Image.open(file_path)
            ocr_text = pytesseract.image_to_string(image)
            if ocr_text:
                pages.append({"page_number": 1, "content": ocr_text})

        elif file_extension == '.xml':
            tree = ET.parse(file_path)
            root = tree.getroot()
            xml_text = " ".join(root.itertext())
            cleaned_text = " ".join(xml_text.split())
            if cleaned_text:
                pages.append({"page_number": 1, "content": cleaned_text})

        else:
            print(f"   - Warning: Unsupported file type '{file_extension}'. Skipping file.")
            return []

    except Exception as e:
        print(f"   - Error processing file {file_path}: {e}")
        return []

    print(f"   + Extracted {len(pages)} page(s) of content.")
    return pages

def chunk_document(pages: list[dict], chunk_size: int = 1000, chunk_overlap: int = 150) -> list[dict]:
    """Splits the document content from pages into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )

    chunks = []
    for page in pages:
        page_content = page.get('content', '')
        split_texts = text_splitter.split_text(page_content)
        for text in split_texts:
            chunks.append({
                "page_number": page.get('page_number'),
                "chunk_text": text
            })
    print(f"   + Document chunked into {len(chunks)} pieces.")
    return chunks

def process_and_upsert_document(doc_path: str, index, model, batch_size: int):
    """Performs the full ETL pipeline for a single document."""
    try:
        pages = extract_text_from_document(doc_path)
        if not pages:
            print(f"   - No content extracted from {doc_path}. Skipping.")
            return

        temp_chunks = chunk_document(pages)
        if not temp_chunks:
            print(f"   - No chunks were created from {doc_path}. Skipping.")
            return

        file_name = Path(doc_path).name
        chunks = []
        for i, chunk in enumerate(temp_chunks):
             chunks.append({
                "chunk_id": f"{file_name}_p{chunk['page_number']}_c{i}",
                "page_number": chunk['page_number'],
                "chunk_text": chunk['chunk_text']
             })

        print(f"   + Uploading {len(chunks)} chunks to Pinecone index '{index_name}'...")
        for i in tqdm(range(0, len(chunks), batch_size), desc=f"Upserting {file_name}", leave=False):
            i_end = min(i + batch_size, len(chunks))
            batch = chunks[i:i_end]

            ids = [chunk['chunk_id'] for chunk in batch]
            texts_to_embed = [chunk['chunk_text'] for chunk in batch]
            metadata = [
                {
                    'page_number': chunk['page_number'],
                    'text': chunk['chunk_text'],
                    'source_file': file_name
                 }
                for chunk in batch
            ]

            embeddings = model.encode(texts_to_embed, normalize_embeddings=True, show_progress_bar=False)
            vectors_to_upsert = list(zip(ids, embeddings.tolist(), metadata))
            index.upsert(vectors=vectors_to_upsert)

        print(f"--- Finished processing {file_name} ---")

    except Exception as e:
        print(f"An unexpected error occurred while processing {doc_path}: {e}")

### 5. Ingestion Process

This section runs the main processing loop to find all documents in your source folder and ingest them into Pinecone.

### 6. RAG Querying and Answering

In [ ]:
# --- Initialize Gemini Model for Generation ---
try:
    generation_config = {
        "temperature": 0.2,
        "top_p": 1,
        "top_k": 5,
        "max_output_tokens": 2048,
    }
    # Note: Adjust the model name if you have access to other models like "gemini-1.5-pro-latest"
    gemini_model = genai.GenerativeModel(
        model_name="gemini-2.5-pro",
        generation_config=generation_config
    )
    print("Gemini model initialized successfully.")
except Exception as e:
    print(f"Error initializing Gemini model: {e}")

In [ ]:
def retrieve_context(query: str, top_k: int = 10) -> str:
    """Retrieves the most relevant context from Pinecone based on a user query."""
    print("Encoding query...")
    query_embedding = model.encode(query, normalize_embeddings=True).tolist()

    print("Querying Pinecone index...")
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    context_parts = []
    if results['matches']:
        print(f"Retrieved {len(results['matches'])} relevant chunks.")
        for match in results['matches']:
            if 'text' in match['metadata']:
                context_parts.append(match['metadata']['text'])
    else:
        print("No relevant context found in the vector database.")

    return "\n\n---\n\n".join(context_parts)

def generate_answer(query: str, context: str) -> str:
    """Generates an answer using the Gemini API based on the query and retrieved context."""
    if not context:
        return "I could not find any relevant information in the documents to answer your question."

    prompt = f"""You are a helpful and informative question-answering assistant.
    Your task is to answer the user's query based ONLY on the provided context.
    Do not use any external knowledge. Cite the source file if it is provided in the context.
    If the answer is not present in the context, state that you cannot answer the question with the given information.

    CONTEXT:
    ---
    {context}
    ---

    QUERY: {query}

    ANSWER:"""

    print("\nGenerating answer with Gemini...")
    try:
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"An error occurred while calling the Gemini API: {e}")
        return "Sorry, I encountered an error while trying to generate an answer."

def ask_question(query: str):
    """Full RAG pipeline: retrieves context, generates answer, and prints results."""
    retrieved_context = retrieve_context(query)

    if retrieved_context:
      print("\n--- Retrieved Context ---")
      print(retrieved_context)
      print("\n-------------------------")

    final_answer = generate_answer(query, retrieved_context)

    print("\n--- Final Answer ---")
    print(final_answer)
    print("--------------------\n")
    return final_answer

In [ ]:
query = "when was the latest Annual General Meeting (AGM) held?"
ask_question(query)

In [ ]:
import csv
import pandas as pd

In [ ]:
questions_to_process = [
    "Form Name",
    "Language",
    "Corporate Identity Number (CIN) of company",
    "Name of the company",
    "Address of registered office of the company",
    "e-mail ID of the company",
    "Authorised capital of the company as on the date of filing (in Rs.)",
    "Number of members of the company as on the date of filing",
    "Type of Industry",
    "Financial year to which financial statement relates: From",
    "Financial year to which financial statement relates: To",
    "Nature of financial statements",
    "Date of Board of Director's meeting in which financial statements are approved",
    "Whether provisional financial statements filed earlier",
    "Whether adopted in adjourned AGM",
    "Whether Annual General Meeting (AGM) held",
    "Date of AGM",
    "Due date of AGM",
    "Whether any extension for financial year or AGM granted",
    "Whether Schedule III of the Companies Act, 2013 is applicable",
    "Whether financial statements have been drawn on the basis of",
    "Whether consolidated Financial Statements are also being filed",
    "Whether CAG of India has commented upon or supplemented the audit report",
    "Whether CAG of India has conducted supplementary or test audit",
    "Whether Secretarial Audit is applicable",
    "Whether detailed disclosure with respect to Director's report Sec 134(3) is attached",
    "XBRL document in respect Consolidated financial statement: Status",
    "List of Attachment(s)",
    "Declaration Resolution number",
    "Declaration Dated",
    "Signatory Designation",
    "Director identification number",
    "Certifying Professional",
    "Professional Status",
    "Membership Number"
]

In [ ]:
wh_question_list = [
    "What is the name of the form being filed?",
    "What is the Corporate Identity Number (CIN) of the company?",
    "What is the legal name of the company?",
    "Where is the registered office of the company located?",
    "What is the official email ID of the company?",
    "What is the authorised capital of the company in Rupees as of the filing date?",
    "How many members does the company have?",
    "What type of industry does the company belong to?",
    "Which financial year do these financial statements cover?",
    "When did the financial year begin and end?",
    "What is the nature of the financial statements (e.g., standalone, consolidated)?",
    "When did the Board of Directors' meeting to approve the financial statements take place?",
    "When was the Annual General Meeting (AGM) held?",
    "What was the due date for the AGM?",
    "Which accounting standards were the financial statements based on?",
    "Who commented on or supplemented the audit report (e.g., CAG of India)?",
    "What disclosures from the Director's report are attached?",
    "What is the status of the XBRL document for the consolidated financial statements?",
    "What attachments are included with this filing?",
    "What is the declaration's resolution number?",
    "When was the declaration dated?",
    "Who signed this document on behalf of the company?",
    "What is the designation or title of the signatory?",
    "What is the Director Identification Number (DIN) of the signatory?",
    "Who is the professional that certified this document?",
    "What is the professional status of the certifier (e.g., Chartered Accountant, Company Secretary)?",
    "What is the membership number of the certifying professional?"
]

In [ ]:
expected_answers = [
    "The Corporate Identity Number (CIN) is U00000XX0000PTC000000.",
    "The authorized capital of the company is 5,000,000.0 Rs.",
    "The Annual General Meeting (AGM) was held on 30/09/2024.",
    "The registered office address is PLOT NO.1, EXAMPLE INDUSTRIAL ESTATE, EXAMPLE, EXAMPLE CITY, Karnataka, 560000, India.",
    "Yes, a Secretarial Audit is applicable.",
    "The financial year covered is from 01/04/2023 to 31/03/2024.",
    "The certifying professional is a Chartered accountant (in whole-time practice) with the status of a Fellow."
]

In [ ]:
def append_to_csv(file_path, data):
    """
    Appends a list of lists to an existing CSV file without overwriting it.

    This is useful for logging data or adding new entries over time.

    Args:
        file_path (str): The path to the CSV file to append to.
        data (list): A list of lists representing the new rows to add.
    """
    print(f"Appending data to '{file_path}'...")
    try:
        # Use 'a' mode to append to the file.
        with open(file_path, 'a', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerows(data)
        print(f"Successfully appended data to '{file_path}'.\n")
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' does not exist. Please create it first.\n")
    except IOError as e:
        print(f"Error appending to {file_path}: {e}\n")

def initialise_csv(file_path):
    print(f"Initialising '{file_path}'...")
    try:
        # Use 'a' mode to append to the file.
        with open(file_path, 'a', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writecolumns(data)
        print(f"Successfully appended data to '{file_path}'.\n")
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' does not exist. Please create it first.\n")
    except IOError as e:
        print(f"Error appending to {file_path}: {e}\n")

In [ ]:
def iterative_question_handler(queries:List, output_file_path:str, df:pd.DataFrame):
  for i in queries:
    count = 0
    ans = input(f"Do you wish to ask {i}? (Y/n)")
    if ans in ["y","Y",""," "]:
      answer = ask_question(i) 
      df = pd.DataFrame([{"Questions":i, "Expected": expected_answers[count],"Recieved": answer}])
      df.to_csv(file_path, mode='a', header=False, index=False)

          




In [ ]:
# Sequential calls for all wh_questions
def process_all_questions(questions: List[str], output_csv_path: str):
    """
    Sequentially processes all questions and saves results to CSV.
    
    Args:
        questions: List of questions to ask
        output_csv_path: Path to save the CSV results
    """
    print(f"Processing {len(questions)} questions sequentially...\n")
    
    # Initialize results list
    results = []
    
    # Process each question sequentially
    for idx, question in enumerate(questions, 1):
        print(f"\n{'='*80}")
        print(f"Question {idx}/{len(questions)}: {question}")
        print(f"{'='*80}")
        
        try:
            # Make the API call and get answer
            answer = ask_question(question)
            
            # Store result
            results.append({
                "Question_Number": idx,
                "Question": question,
                "Answer": answer
            })
            
        except Exception as e:
            print(f"Error processing question {idx}: {e}")
            results.append({
                "Question_Number": idx,
                "Question": question,
                "Answer": f"ERROR: {str(e)}"
            })
    
    # Save all results to CSV
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv_path, index=False)
    print(f"\n{'='*80}")
    print(f"All {len(questions)} questions processed!")
    print(f"Results saved to: {output_csv_path}")
    print(f"{'='*80}")
    
    return df_results